In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/fadhilr2/ml_data/refs/heads/main/hoax_dataset.csv")
df = df[:1000]

In [ ]:
df.info()

In [ ]:
from transformers import pipeline

pipe = pipeline("feature-extraction",
                model="indobenchmark/indobert-base-p1")

In [ ]:
from transformers import pipeline

ner_pipe = pipeline("token-classification",
                model="cahya/bert-base-indonesian-NER",
                aggregation_strategy="simple")




In [ ]:
import requests
import json
import os


def searchWikiData(word):
  url = "https://www.wikidata.org/w/api.php"
  params = {
    "action": "wbsearchentities",
    "search": word,
    "language": "id",
    "format": "json"
  }
  headers = {
    "User-Agent": "KAPALM_KG_Builder/1.0 (fadhilra08@gmail.com)"
  }

  try:
    response = requests.get(url, headers=headers, params=params)

    response.raise_for_status()

    data = response.json()

    return data

  except requests.exceptions.RequestException as e:
      return None

CACHE_FILE = "wikidata_cache.json"
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        wiki_cache = json.load(f)
else:
    wiki_cache = {}

def searchWikiData_Cached(word):
    if word in wiki_cache:
        return wiki_cache[word]

    data = searchWikiData(word)

    if data:
        wiki_cache[word] = data
        if len(wiki_cache) % 50 == 0:
            with open(CACHE_FILE, "w") as f:
                json.dump(wiki_cache, f)
    return data


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

def getWikidataKG(entity_id):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.agent = "KAPALM_KG_Builder/1.0 (fadhilra08@gmail.com)"

    query = f"""
      SELECT ?propUrl ?propertyLabel ?object ?objectLabel WHERE {{
      VALUES ?propUrl {{ wdt:P31 wdt:P279 wdt:P17 wdt:P131 wdt:P361 wdt:P106 wdt:P69 wdt:P452 }}
      wd:{entity_id} ?propUrl ?object .
      FILTER(isIRI(?object))
      ?property wikibase:directClaim ?propUrl .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "id". }}
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    try:
        results = sparql.query().convert()
    except Exception as e:
        print(f"SPARQL Error {entity_id}: {e}")
        return []

    triplets = []
    for result in results["results"]["bindings"]:
        subject_id = entity_id

        predicate_code = result["propUrl"]["value"].split('/')[-1]

        predicate_label = result.get("propertyLabel", {}).get("value", predicate_code)

        if result["object"]["type"] == "uri":
            obj_id = result["object"]["value"].split('/')[-1]
            obj_label = result.get("objectLabel", {}).get("value", obj_id)

            triplets.append({
                "subject_id": subject_id,
                "predicate_code": predicate_code,
                "predicate_label": predicate_label,
                "object_id": obj_id,
                "object_label": obj_label
            })
    return triplets

In [ ]:
import networkx as nx

def generateGraph(triplets, subject_label):
    G = nx.MultiDiGraph()
    for e in triplets:
        if not G.has_node(e["subject_id"]):
            G.add_node(e["subject_id"], label=subject_label)
        if not G.has_node(e["object_id"]):
            G.add_node(e["object_id"], label=e["object_label"])

        G.add_edge(
            e["subject_id"],
            e["object_id"],
            relation_code=e["predicate_code"], # Use this for mapping
            relation_label=e["predicate_label"]
        )
    return G

In [ ]:
def retrieveKg(extracted_entities):
    G = nx.MultiDiGraph()

    for ent in extracted_entities:
        word = ent["word"]
        data = searchWikiData_Cached(word)

        if data and "search" in data and len(data["search"]) > 0:
            top_result = data["search"][0]
            entity_id = top_result["id"]

            official_label = top_result.get("label", word)

            triplets = getWikidataKG(entity_id)

            KG = generateGraph(triplets, official_label)
            G = nx.compose(G, KG)

    return G

In [ ]:
from torch_geometric.data import Data

def networkXtoData(G, embedding_model, tokenizer, global_rel_map):
    node_list = sorted(list(G.nodes(data=True)))
    node_to_idx = {node[0]: i for i, node in enumerate(node_list)}
    labels = [attr.get('label', str(n_id)) for n_id, attr in node_list]

    with torch.no_grad():
        inputs = tokenizer(labels, padding=True, truncation=True, return_tensors="pt")
        outputs = embedding_model(**inputs)
        node_features = outputs.last_hidden_state[:, 0, :]

    src_list = []
    dst_list = []
    edge_type_list = []

    for src, dst, attr in G.edges(data=True):
        rel_code = attr.get('relation_code', 'related_to')

        if rel_code in global_rel_map:
            src_list.append(node_to_idx[src])
            dst_list.append(node_to_idx[dst])
            edge_type_list.append(global_rel_map[rel_code])
        else:
            src_list.append(node_to_idx[src])
            dst_list.append(node_to_idx[dst])
            edge_type_list.append(global_rel_map['related_to'])

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    edge_type = torch.tensor(edge_type_list, dtype=torch.long)

    return Data(x=node_features, edge_index=edge_index, edge_type=edge_type)

In [ ]:
from transformers import AutoTokenizer, AutoModel


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
embedding_model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

In [ ]:
def extract_entities_from_text(text):
  results = ner_pipe(text[:512])

  entities = []

  valid_labels = ["PER", "ORG", "LOC", "GPE"]

  for item in results:
    if item["entity_group"] in valid_labels:
      entities.append({
          "word": item["word"],
          "label": item["entity_group"]
      })
  return entities

In [ ]:
import time

In [ ]:

processed_data = []

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
  text = row["text"]
  label = row["label"]

  entities = extract_entities_from_text(text)

  nx_graph = retrieveKg(entities)
  if nx_graph.number_of_nodes() == 0:
        nx_graph.add_node("dummy", label="tidak_diketahui")
        nx_graph.add_edge("dummy", "dummy", relation="related_to")

  pyg_data = networkXtoData(nx_graph, embedding_model, tokenizer, global_rel_map)

  processed_data.append({
      "text": text,
      "graph": pyg_data,
      "label": label
  })

  time.sleep(15)

torch.save(processed_data, 'kapalm_indonesian_data.pt')
print("Done! Saved to kapalm_indonesian_data.pt")
